# Phase 02: Dataset Preprocessing & Formatting - Code Mentor AI

In this notebook, we take the raw **`ise-uiuc/Magicoder-Evol-Instruct-110K`** dataset and prepare it for Supervised Fine-Tuning (SFT). 

### Notebook Objectives:
1. Load dataset & inspect structure.
2. Clean data (handle missing, empty, or malformed examples).
3. Remove duplicate instructions/responses.
4. Convert training examples into a conversational chat format (System + User + Assistant).
5. Create Train, Validation, and Test splits.
6. Save the processed dataset to disk.

In [1]:
#libraries 
import os
import pandas as pd
import matplotlib.pyplot as plt
from datasets import DatasetDict
from datasets import load_dataset

In [2]:
# Step 1: Environment Setup & Dataset Loading
print("=" * 60)
print("Loading Magicoder dataset for preprocessing...")
print("=" * 60)

ds = load_dataset("ise-uiuc/Magicoder-Evol-Instruct-110K")
train_data = ds['train']

print("=" * 60)
print(f"Initial Total Rows: {len(train_data):,}")
print(f"Columns: {train_data.column_names}")

Loading Magicoder dataset for preprocessing...


README.md:   0%|          | 0.00/394 [00:00<?, ?B/s]

data-evol_instruct-decontaminated.jsonl: reconstructing file:   0%|          |  0.00B /  255MB            

data-evol_instruct-decontaminated.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/111183 [00:00<?, ? examples/s]

Initial Total Rows: 111,183
Columns: ['instruction', 'response']


### Step 2: Data Cleaning (Missing & Empty Check)
We convert to a pandas DataFrame to easily filter out missing or whitespace-only rows.

In [3]:
# Step 2: Handle Missing, Empty & Malformed Examples
df = train_data.to_pandas()

# Drop rows with null values in instruction or response
initial_count = len(df)
df = df.dropna(subset=['instruction', 'response'])

# Strip whitespaces and drop empty strings
df['instruction'] = df['instruction'].astype(str).str.strip()
df['response'] = df['response'].astype(str).str.strip()

df = df[(df['instruction'] != "") & (df['response'] != "")]

print(f"Rows before cleaning: {initial_count:,}")
print(f"Rows after dropping nulls/empty strings: {len(df):,}")
print(f"Removed rows: {initial_count - len(df):,}")

Rows before cleaning: 111,183
Rows after dropping nulls/empty strings: 111,183
Removed rows: 0


### Step 3: Duplicate Removal
Removing duplicate instruction queries to prevent data leakage and model overfitting on repetitive coding problems.

In [4]:
#Step 3: Remove duplicate instructions
duplicates_before = df.duplicated(subset=['instruction']).sum()
print(f"Duplicate instruction entries found: {duplicates_before:,}")

df = df.drop_duplicates(subset=['instruction']).reset_index(drop=True)
print(f"Total rows after deduplication: {len(df):,}")

Duplicate instruction entries found: 0
Total rows after deduplication: 111,183


### Step 4: Conversational Chat Formatting
Formatting data into a structured chat template (`messages` format) so the model learns a clear **Code Mentor AI** persona, explaining code clearly, debugging errors, and providing structured solutions.

In [5]:
# Step 4: Convert into Conversational Format
def format_to_chat(row):
    system_prompt = (
        "You are Code Mentor AI, an expert software engineer and programming tutor. "
        "Your goal is to help users write clean code, explain complex algorithms step-by-step, "
        "and debug errors thoroughly."
    )
    return {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": row['instruction']},
            {"role": "assistant", "content": row['response']}
        ]
    }

# Apply formatting across the dataframe
print("Formatting rows into chat templates...")
formatted_data = [format_to_chat(row) for _, row in df.iterrows()]

# Convert back to Hugging Face Dataset format
from datasets import Dataset
processed_dataset = Dataset.from_pandas(pd.DataFrame(formatted_data))
print('='*60)
print("Formatting complete! Sample chat structure:")
print(processed_dataset[0])

Formatting rows into chat templates...
Formatting complete! Sample chat structure:
{'messages': [{'content': 'You are Code Mentor AI, an expert software engineer and programming tutor. Your goal is to help users write clean code, explain complex algorithms step-by-step, and debug errors thoroughly.', 'role': 'system'}, {'content': "Please amend the subsequent Python script so that it includes a 'while' loop rather than the existing 'for' loop, which iterates through the items of an integer list.\n\nThe script currently has a bug where it attempts to print an object that is outside the bounds of the list. Fix this error and modify the script to use 'while' instead of 'for' loop. Ensure your script correctly handles empty lists. \n\n```python\n  # Establish an integer list\n  arr = [1, 2, 3, 4]\n\n  # Determine the length of the list\n  n = len(arr)\n\n  # Traverse the list and output each individual element\n  for i in range(n+1):\n      print(arr[i])\n```", 'role': 'user'}, {'content':

### Step 5: Creating Train, Validation & Test Splits
Splitting our cleaned dataset to ensure we have a robust split for training, validation loss monitoring, and final evaluation.

In [6]:
# Step 5: Train / Validation / Test Split
# Split ratio: 90% Train, 5% Validation, 5% Test
train_val_test = processed_dataset.train_test_split(test_size=0.10, seed=42)
test_val = train_val_test['test'].train_test_split(test_size=0.50, seed=42)


final_dataset_split = DatasetDict({
    'train': train_val_test['train'],
    'validation': test_val['train'],
    'test': test_val['test']
})

print("Dataset Splits Created Successfully:")
print(final_dataset_split)

Dataset Splits Created Successfully:
DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 100064
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 5559
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 5560
    })
})


### Step 6: Saving the Processed Dataset
Saving the final processed dataset locally so it can be directly loaded in the Phase 03 fine-tuning script.

In [7]:
# Step 6: Save Processed Dataset to Disk (Kaggle Optimized)
save_directory = "/kaggle/working/processed_code_mentor_data"
final_dataset_split.save_to_disk(save_directory)

print(f"Successfully saved processed dataset to: {save_directory}")

Saving the dataset (0/1 shards):   0%|          | 0/100064 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5559 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5560 [00:00<?, ? examples/s]

Successfully saved processed dataset to: /kaggle/working/processed_code_mentor_data


## Conclusion: Phase 02 Dataset Preprocessing & Formatting
Your second notebook successfully transforms the raw Magicoder dataset into a clean, model-ready format optimized for Supervised Fine-Tuning (SFT) on Kaggle.

## Summary of Achievements
`Data Hygiene:` Filtered out missing, null, and empty entries to ensure a high standard of data quality.

`Deduplication:` Removed duplicate instructions to prevent model memorization and overfitting during training.

`Chat Template Structuring:` Wrapped data into a conversational schema (system, user, assistant) to establish the distinct Code Mentor AI persona.

`Dataset Splitting:` Generated robust train, validation, and test splits (90/5/5 ratio) for proper loss tracking and evaluation.

`Kaggle Optimization:` Successfully persisted the final split dataset to /kaggle/working/processed_code_mentor_data for seamless reloading.